In [22]:
import pandas as pd

df = pd.read_csv("./Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## **Preprocessing**

### **Removing unnecessary columns**

In [23]:


df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [24]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### **Train-Test Split**

In [25]:
X = df.drop(['Exited'], axis=1)
y = df['Exited']

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### **Encoding categorical features and Standardization**

In [27]:
X_train['Geography'].value_counts(), X_test['Geography'].value_counts()

(Geography
 France     3994
 Germany    2011
 Spain      1995
 Name: count, dtype: int64,
 Geography
 France     1020
 Germany     498
 Spain       482
 Name: count, dtype: int64)

In [28]:
X_train['Gender'].value_counts(), X_test['Gender'].value_counts()

(Gender
 Male      4362
 Female    3638
 Name: count, dtype: int64,
 Gender
 Male      1095
 Female     905
 Name: count, dtype: int64)

In [29]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

num_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

ct = ColumnTransformer([
    ('ohe', OneHotEncoder(drop='first'), ['Geography']),
    ('oe', OrdinalEncoder(), ['Gender']),
    ('scaler', StandardScaler(), num_features)
], remainder='passthrough')

X_train_preprocessed = ct.fit_transform(X_train)
X_test_preprocessed = ct.transform(X_test)


In [31]:
X_train_preprocessed[0:5]

array([[ 0.        ,  0.        ,  1.        ,  0.35649971, -0.6557859 ,
         0.34567966, -1.21847056,  0.80843615,  1.36766974,  1.        ,
         1.        ],
       [ 1.        ,  0.        ,  1.        , -0.20389777,  0.29493847,
        -0.3483691 ,  0.69683765,  0.80843615,  1.6612541 ,  1.        ,
         1.        ],
       [ 0.        ,  1.        ,  1.        , -0.96147213, -1.41636539,
        -0.69539349,  0.61862909, -0.91668767, -0.25280688,  1.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        , -0.94071667, -1.13114808,
         1.38675281,  0.95321202, -0.91668767,  0.91539272,  1.        ,
         0.        ],
       [ 0.        ,  0.        ,  1.        , -1.39733684,  1.62595257,
         1.38675281,  1.05744869, -0.91668767, -1.05960019,  0.        ,
         0.        ]])

### **Saving the preprocessor object**

In [32]:
import pickle

with open('preprocessor.pkl', 'wb') as f:
    pickle.dump(ct, f)

## **ANN Implementation**

In [33]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

### **Building ANN model**

In [43]:
model = Sequential(
    [
        Dense(64, activation='relu', input_shape=(X_train_preprocessed.shape[1],)),    # Hidden layer 1
        Dense(32, activation='relu'),       # Hidden Layer 2
        Dense(1, activation='sigmoid')      # Output layer
    ]
)

In [44]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

### **Compiling model**

In [45]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)

In [46]:
model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy', 'auc'])

### **Setup the Tensorboard**

In [47]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [48]:
## >>> Early stopping <<<

early_stop_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

### **Training the model**

In [49]:
history = model.fit(X_train_preprocessed, y_train, validation_data=(X_test_preprocessed, y_test), epochs=100, callbacks=[early_stop_callback, tensorflow_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8241 - auc: 0.8021 - loss: 0.4014 - val_accuracy: 0.8515 - val_auc: 0.8438 - val_loss: 0.3563
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8559 - auc: 0.8433 - loss: 0.3589 - val_accuracy: 0.8615 - val_auc: 0.8529 - val_loss: 0.3533
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8560 - auc: 0.8521 - loss: 0.3512 - val_accuracy: 0.8615 - val_auc: 0.8572 - val_loss: 0.3411
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8590 - auc: 0.8576 - loss: 0.3445 - val_accuracy: 0.8610 - val_auc: 0.8595 - val_loss: 0.3362
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8611 - auc: 0.8623 - loss: 0.3397 - val_accuracy: 0.8585 - val_auc: 0.8529 - val_loss: 0.3493
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8596 - auc: 0.8632 - loss: 0.3393 - val_accuracy: 0.8665 - val_auc: 0.8556 - val_loss: 0.3380
Epoch 7/100
250/250 ━━━━━━━━━━━━━━

### **Saving the model**

In [51]:
model.save('model.keras')